# PM-RL1 路线选择：离线复查
这里只读取派生汇总；完整原始绑定检查见 account_routes.py。无网络、模型调用、标签解密或训练。
费用为情景；dev6尚未选定，test合法终点总数未知。

In [1]:
from pathlib import Path
import csv, json
root = Path('/home/tokkio/Metacom5_3/project/outputs/pm_rl1/route_accounting_20261014_v1')
a = root / 'analysis_final'
with (a / 'prefix_head_coverage.csv').open(encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))
totals = {h: sum(int(r['terminals']) for r in rows if r['head'] == h) for h in ['RS','MP','MS','ME']}
assert totals == {'RS': 570, 'MP': 488, 'MS': 360, 'ME': 563}
assert max(totals, key=totals.get) == 'RS'
print('按冻结合法剂量覆盖选头：', totals)
with (a / 'b1_terminal_reuse_private.csv').open(encoding='utf-8-sig') as f:
    terminals = list(csv.DictReader(f))
train = [r for r in terminals if r['split'] == 'train']
assert len(train) == 120
assert sum(r['reusable'] == 'True' for r in train) == 72
dev = [r for r in terminals if r['split'] == 'dev']
assert len(dev) == 90  # 18个候选前缀；不是已选dev6
assert all(sum(r['reusable'] == 'True' for r in dev if r['index'] == i) == 3 for i in {r['index'] for r in dev})
print('train：120终点、72可复用、48缺失；任何合法dev6：30/18/12。')
b = json.loads((a / 'alternative_budget.json').read_text())
assert b['existing_P3_starts'] == 66 and b['unchanged_actual_request_cap'] == 140
assert 66 + 33 + 47 + 12 == 158
assert 158 + 4 == 162
with (a / 'alternative_trial_budget.csv').open(encoding='utf-8-sig') as f:
    costs = list(csv.DictReader(f))
lo = sum(float(r['peak_usd_estimate_low']) for r in costs)
hi = sum(float(r['peak_usd_estimate_high']) for r in costs)
assert abs(lo - b['estimated_trial_peak_usd'][0]) < 1e-6
assert abs(hi - b['estimated_trial_peak_usd'][1]) < 1e-6
print('96次试验预算情景（不是授权或账单）：', round(lo,6), round(hi,6), '美元')
print('保持未知：test真实mask、供应商真实质量/吞吐/用量、人工计时和裁决数。')


按冻结合法剂量覆盖选头： {'RS': 570, 'MP': 488, 'MS': 360, 'ME': 563}
train：120终点、72可复用、48缺失；任何合法dev6：30/18/12。
96次试验预算情景（不是授权或账单）： 0.592931 1.402518 美元
保持未知：test真实mask、供应商真实质量/吞吐/用量、人工计时和裁决数。
